# YOLO Fruit Detection Training - Kaggle Version

Train YOLO v8 model for multi-class fruit detection (Mango, Grapefruit, Guava, Orange)

**Hardware Requirements:**
- GPU: P100 or better
- RAM: 16GB+ recommended

**Dataset:**
- Add your dataset as Kaggle Dataset input
- Path will be: `/kaggle/input/your-dataset-name/`

In [ ]:
# Install Required Packages for Kaggle
!pip install ultralytics

print("✅ Packages installed successfully!")

In [ ]:
# Import Required Libraries
import os
import torch
from ultralytics import YOLO
import yaml
from pathlib import Path

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# Check Kaggle Environment and Dataset
print("🔍 Kaggle Environment Check:")
print(f"Working directory: {os.getcwd()}")
print(f"Available datasets: {os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'No input datasets'}")

# List all files in input directory to find your dataset
if os.path.exists('/kaggle/input'):
    for item in os.listdir('/kaggle/input'):
        item_path = f'/kaggle/input/{item}'
        if os.path.isdir(item_path):
            print(f"📁 Dataset: {item}")
            print(f"   Contents: {os.listdir(item_path)[:10]}...")  # Show first 10 items

In [ ]:
# Verify Dataset Structure
# ⚠️ IMPORTANT: Replace 'your-dataset-name' with your actual Kaggle dataset name
# Example: If your dataset is named 'multi-fruit-detection', use that name

DATASET_NAME = "your-dataset-name"  # 🔄 CHANGE THIS to your uploaded dataset name
dataset_path = Path(f'/kaggle/input/{DATASET_NAME}/multi_fruit_detection')

print(f"🔍 Looking for dataset at: {dataset_path}")

# If dataset is directly in input folder (not in subfolder)
if not dataset_path.exists():
    dataset_path = Path(f'/kaggle/input/{DATASET_NAME}')
    print(f"🔍 Trying alternative path: {dataset_path}")

# Quick check for required structure
if not dataset_path.exists():
    print("❌ Dataset not found!")
    print("Please check:")
    print("1. Dataset is uploaded to Kaggle")
    print("2. Dataset is added as input to this notebook")
    print("3. DATASET_NAME variable above matches your dataset name")
else:
    print("✅ Dataset directory found!")
    
    required_dirs = ['train/images', 'train/labels', 'val/images', 'val/labels']
    all_exist = all((dataset_path / d).exists() for d in required_dirs)
    
    if all_exist:
        print("✅ All required directories present!")
        
        # Count images for verification
        train_images = len(list((dataset_path / 'train/images').glob('*.jpg')))
        val_images = len(list((dataset_path / 'val/images').glob('*.jpg')))
        
        print(f"📊 Dataset Summary:")
        print(f"   Train images: {train_images}")
        print(f"   Val images: {val_images}")
        print(f"   Total: {train_images + val_images}")
    else:
        print("⚠️ Some required directories are missing!")
        for d in required_dirs:
            status = "✅" if (dataset_path / d).exists() else "❌"
            print(f"{status} {d}")

In [ ]:
# Create data.yaml Configuration
yaml_content = {
    'path': str(dataset_path),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 4,
    'names': ['mango', 'grapefruit', 'guava', 'orange']
}

# Save yaml file to working directory (Kaggle allows writing here)
yaml_file = Path('/kaggle/working/data.yaml')
with open(yaml_file, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("✅ Created data.yaml configuration")
print(f"📄 YAML file: {yaml_file}")
print(f"Classes: {yaml_content['nc']}")
print(f"Names: {yaml_content['names']}")

# Display the created yaml content
print("\n📋 YAML Configuration:")
with open(yaml_file, 'r') as f:
    print(f.read())

In [ ]:
# Initialize YOLO Model
model = YOLO('yolov8n.pt')
print("✅ YOLOv8 Nano model loaded successfully!")
print("📦 Model will be downloaded automatically on first run")

In [ ]:
# Train YOLO Model
print("🚀 Starting YOLO training on Kaggle...")
print("⏱️ Estimated time: 3-5 hours on P100 GPU")
print("📊 Training progress will be displayed below")

# Training with optimized parameters for Kaggle
results = model.train(
    data=str(yaml_file),
    epochs=100,
    imgsz=640,
    batch=32,                    # Larger batch size for P100 GPU
    name='multi_fruit_detection_v1',
    patience=10,
    save_period=10,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True,
    plots=True,
    workers=4,                   # Optimize data loading
    project='/kaggle/working/runs/detect'  # Save to working directory
)

print("🎉 Training completed!")
print("📁 Results saved in: /kaggle/working/runs/detect/multi_fruit_detection_v1/")

In [ ]:
# Evaluate Model - Comprehensive Results
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Load best model
best_model_path = '/kaggle/working/runs/detect/multi_fruit_detection_v1/weights/best.pt'
best_model = YOLO(best_model_path)
print("✅ Best model loaded\n")

# Evaluate on validation set
metrics = best_model.val()

# Display comprehensive performance metrics
print("="*60)
print("📊 MODEL PERFORMANCE METRICS")
print("="*60)
print(f"mAP@0.5:        {metrics.box.map50:.4f} {'✅ Excellent' if metrics.box.map50 > 0.85 else '⚠️ Needs improvement' if metrics.box.map50 > 0.70 else '❌ Poor'}")
print(f"mAP@0.5-0.95:   {metrics.box.map:.4f} {'✅ Excellent' if metrics.box.map > 0.70 else '⚠️ Good' if metrics.box.map > 0.50 else '❌ Needs improvement'}")
print(f"Precision:      {metrics.box.mp:.4f} {'✅ High' if metrics.box.mp > 0.85 else '⚠️ Moderate' if metrics.box.mp > 0.70 else '❌ Low'}")
print(f"Recall:         {metrics.box.mr:.4f} {'✅ High' if metrics.box.mr > 0.85 else '⚠️ Moderate' if metrics.box.mr > 0.70 else '❌ Low'}")
print("="*60)

# Performance assessment
print("\n🎯 TRAINING QUALITY ASSESSMENT:")
if metrics.box.map50 > 0.85 and metrics.box.map > 0.65:
    print("✅ Model is WELL-TRAINED and ready for deployment!")
elif metrics.box.map50 > 0.70:
    print("⚠️ Model shows GOOD performance but could be improved")
else:
    print("❌ Model needs MORE TRAINING or data improvement")

# Display per-class metrics if available
print("\n📋 PER-CLASS PERFORMANCE:")
class_names = ['mango', 'grapefruit', 'guava', 'orange']
if hasattr(metrics.box, 'ap50'):
    for i, name in enumerate(class_names):
        if i < len(metrics.box.ap50):
            ap = metrics.box.ap50[i]
            print(f"{name:12s}: {ap:.4f}")

# Display all available training visualizations
print("\n📈 TRAINING VISUALIZATIONS:")
results_dir = '/kaggle/working/runs/detect/multi_fruit_detection_v1'

# 1. Training curves (Loss, mAP, Precision, Recall)
if os.path.exists(f'{results_dir}/results.png'):
    print("\n1️⃣ Training Curves (Loss, Metrics over Epochs):")
    display(Image(f'{results_dir}/results.png'))

# 2. Confusion Matrix
if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    print("\n2️⃣ Confusion Matrix:")
    display(Image(f'{results_dir}/confusion_matrix.png'))

# 3. F1 Curve
if os.path.exists(f'{results_dir}/F1_curve.png'):
    print("\n3️⃣ F1-Confidence Curve:")
    display(Image(f'{results_dir}/F1_curve.png'))

# 4. Precision-Recall Curve
if os.path.exists(f'{results_dir}/PR_curve.png'):
    print("\n4️⃣ Precision-Recall Curve:")
    display(Image(f'{results_dir}/PR_curve.png'))

# 5. Validation Batch Labels
if os.path.exists(f'{results_dir}/val_batch0_labels.jpg'):
    print("\n5️⃣ Validation Batch - Ground Truth Labels:")
    display(Image(f'{results_dir}/val_batch0_labels.jpg'))

# 6. Validation Batch Predictions
if os.path.exists(f'{results_dir}/val_batch0_pred.jpg'):
    print("\n6️⃣ Validation Batch - Model Predictions:")
    display(Image(f'{results_dir}/val_batch0_pred.jpg'))

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE - Review the metrics and visualizations above")
print("📁 Download trained model from: /kaggle/working/runs/detect/multi_fruit_detection_v1/weights/")
print("="*60)

In [ ]:
# Download Model Files (Optional)
# Kaggle automatically saves output files, but you can also manually save important files

import shutil

# Copy important files to output directory for easy download
output_dir = '/kaggle/working/model_output'
os.makedirs(output_dir, exist_ok=True)

# Copy model weights
if os.path.exists(f'{results_dir}/weights/best.pt'):
    shutil.copy(f'{results_dir}/weights/best.pt', f'{output_dir}/best_model.pt')
    print("✅ Best model copied to output")

# Copy training results
if os.path.exists(f'{results_dir}/results.png'):
    shutil.copy(f'{results_dir}/results.png', f'{output_dir}/training_results.png')
    print("✅ Training results copied to output")

# Copy confusion matrix
if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    shutil.copy(f'{results_dir}/confusion_matrix.png', f'{output_dir}/confusion_matrix.png')
    print("✅ Confusion matrix copied to output")

print(f"\n📁 Output files saved to: {output_dir}")
print("📥 These files will be available for download after notebook execution")

# Show final summary
print("\n🎯 TRAINING COMPLETE SUMMARY:")
print(f"   Model: YOLOv8 Nano")
print(f"   Classes: 4 (mango, grapefruit, guava, orange)")
print(f"   Final mAP@0.5: {metrics.box.map50:.4f}")
print(f"   Training Platform: Kaggle")
print("   Status: ✅ Complete")